In [6]:
import pandas as pd
import numpy as np

SEED = 7

df = pd.read_csv("06_diving.csv")

# (1)/(2) compute each event's mean judge score, then the per-row difference from that mean
df['EventMeanScore'] = df.groupby('Event')['JScore'].transform('mean')
df['DiffFromMean'] = df['JScore'] - df['EventMeanScore']

# (4) flag whether the judge shares the diver's country
df['SameCountry'] = df['Country'] == df['JCountry']

results = []

for judge, jdf in df.groupby('Judge'):
    set1 = jdf.loc[jdf['SameCountry'], 'DiffFromMean'].to_numpy()   # same country
    set2 = jdf.loc[~jdf['SameCountry'], 'DiffFromMean'].to_numpy()  # different country

    n1, n2 = len(set1), len(set2)

    # need at least one observation in each set to compute a meaningful statistic
    if n1 == 0 or n2 == 0:
        results.append({
            'Judge': judge,
            'n_same_country': n1,
            'n_diff_country': n2,
            'observed_stat': np.nan,
            'p_value': np.nan
        })
        continue

    # (5) observed statistic: mean(set1) - mean(set2)
    observed_stat = set1.mean() - set2.mean()

    # (6) permutation test: pool both sets, repeatedly resplit by cardinality
    pooled = np.concatenate([set1, set2])
    rng = np.random.default_rng(SEED)  # same seed for every judge

    n_perm = 10_000
    perm_stats = np.empty(n_perm)

    for i in range(n_perm):
        permuted = rng.permutation(pooled)
        perm_set1 = permuted[:n1]
        perm_set2 = permuted[n1:n1 + n2]
        perm_stats[i] = perm_set2.mean() - perm_set1.mean()

    # (7) the permutation distribution of differences-of-means is centered at 0
    #     (since under the null, label assignment doesn't matter)

    # (8) p-value: probability of observing a statistic at least as extreme (>=) as the observed one
    p_value = np.mean(perm_stats >= observed_stat)

    results.append({
        'Judge': judge,
        'n_same_country': n1,
        'n_diff_country': n2,
        'observed_stat': observed_stat,
        'p_value': p_value
    })

results_df = pd.DataFrame(results).sort_values('p_value')
print(results_df.to_string(index=False))


#Adjusted p-score using Benjamin-Hochberg Adjustment for Multiple Testing

Q = 0.05

# (1) drop NaNs and sort by ascending p-value
bh_df = results_df.dropna(subset=['p_value']).sort_values('p_value').reset_index(drop=True)

m = len(bh_df)
bh_df['rank_k'] = np.arange(1, m + 1)

# (2) BH critical value/score for each entry: (k/m) * Q
bh_df['score'] = (bh_df['rank_k'] / m) * Q

# (3) find the largest k where p_value <= score; everything up to and including
# that entry is statistically significant
significant_mask = bh_df['p_value'] <= bh_df['score']

if significant_mask.any():
    K = bh_df.index[significant_mask].max()
    significant_df = bh_df.loc[:K, ['Judge', 'p_value', 'score']]
else:
    significant_df = bh_df.iloc[0:0][['Judge', 'p_value', 'score']]

# (4) table of score and p-value for the significant entries
print(significant_df.to_string(index=False))


#Nothing changed! Results stayed the same.

                  Judge  n_same_country  n_diff_country  observed_stat  p_value
           ZAITSEV Oleg              38             557       1.280186   0.0000
           WANG Facheng              22             335       1.473974   0.0000
        McFARLAND Steve              42             615       1.100527   0.0000
              XU Yiming              18             263       1.977198   0.0000
             ALT Walter              25             473       1.152338   0.0005
           SEAMAN Kathy              16             265       1.099208   0.0019
      BARNETT Madeleine              38             623       0.800658   0.0020
          BOYS Beverley              13             398       0.702745   0.0514
        BURK Hans-Peter              10             149       0.510738   0.1406
             MENA Jesus              28             828       0.234478   0.2105
       BOOTHROYD Sydney              16             395       0.194113   0.3170
            HUBER Peter               8 

In [9]:
#Q2
import pandas as pd
import statsmodels.formula.api as smf

df = pd.read_csv("06_tto.csv")

# build the TTO indicator variables (named without spaces/symbols for the formula API)
df['TTO2'] = (df['ORDER_CT'] >= 2).astype(int)
df['TTO3'] = (df['ORDER_CT'] >= 3).astype(int)

# ----- Model 1: no continuous t_i term, just the 2TTO/3TTO indicators -----
model1 = smf.ols(
    formula="EVENT_WOBA_19 ~ TTO2 + TTO3 + WOBA_FINAL_BAT_19 + WOBA_FINAL_PIT_19 + HAND_MATCH + BAT_HOME_IND",
    data=df
).fit()

print("=" * 70)
print("MODEL 1")
print("=" * 70)
print(model1.summary())

print("\nModel 1 coefficients and p-values:")
print(pd.DataFrame({
    'coef': model1.params,
    'p_value': model1.pvalues
}))

# ----- Model 2: includes continuous t_i (ORDER_CT) plus the TTO indicators -----
model2 = smf.ols(
    formula="EVENT_WOBA_19 ~ BATTER_SEQ_NUM + TTO2 + TTO3 + WOBA_FINAL_BAT_19 + WOBA_FINAL_PIT_19 + HAND_MATCH + BAT_HOME_IND",
    data=df
).fit()

print("\n" + "=" * 70)
print("MODEL 2")
print("=" * 70)
print(model2.summary())

print("\nModel 2 coefficients and p-values:")
print(pd.DataFrame({
    'coef': model2.params,
    'p_value': model2.pvalues
}))


#Order count reveals that the coefficient results for TTO2 and TTO3 are not significant - thus, we cannot reject the null, and we should assume TTO2 and TTO3 are 0. 

MODEL 1
                            OLS Regression Results                            
Dep. Variable:          EVENT_WOBA_19   R-squared:                       0.022
Model:                            OLS   Adj. R-squared:                  0.022
Method:                 Least Squares   F-statistic:                     811.3
Date:                Mon, 08 Jun 2026   Prob (F-statistic):               0.00
Time:                        13:25:14   Log-Likelihood:            -1.5787e+05
No. Observations:              214386   AIC:                         3.157e+05
Df Residuals:                  214379   BIC:                         3.158e+05
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            -0.2995  